# CardioVision3D — Model Performance & Evaluation Benchmark

**Clinical Decision Intelligence Platform**  
This notebook provides the comprehensive, reproducible evaluation and validation benchmark for all four trained machine learning pipelines:
1. **Overall Coronary Artery Disease (Cath / CAD)**
2. **Left Anterior Descending Artery (LAD Stenosis)**
3. **Left Circumflex Artery (LCX Stenosis)**
4. **Right Coronary Artery (RCA Stenosis)**

Models were trained using 5-fold stratified cross-validation on 80% training data (N=242) and evaluated on an untouched, stratified 20% held-out test cohort (N=61).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Resolve paths
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
METRICS_PATH = ROOT / 'results' / 'metrics.json'
DATA_PATH = ROOT / 'data' / 'raw' / 'extention of Z-Alizadeh sani dataset.xlsx'
MODEL_DIR = ROOT / 'models'

print(f"Project Root: {ROOT}")
print(f"Metrics File Exists: {METRICS_PATH.exists()}")
print(f"Dataset Exists: {DATA_PATH.exists()}")

## 1. Load Evaluated Metrics JSON
Inspect the serialized held-out test set metrics across all four diagnostic targets.

In [ ]:
with open(METRICS_PATH, 'r', encoding='utf-8') as f:
    metrics_data = json.load(f)

summary_rows = []
for target_key, data in metrics_data.items():
    test_m = data['held_out_test_metrics']
    cv_cand = data['cross_validation_candidates'][data['selected_model']]
    summary_rows.append({
        'Target Key': target_key.upper(),
        'Diagnostic Outcome': data['target'],
        'Selected Model': data['selected_model'].replace('_', ' ').title(),
        '5-Fold CV ROC-AUC': f"{cv_cand['cv_roc_auc_mean']:.3f} ± {cv_cand['cv_roc_auc_std']:.3f}",
        'Test Accuracy': f"{test_m['accuracy'] * 100:.1f}%",
        'Test Precision': f"{test_m['precision'] * 100:.1f}%",
        'Test Recall': f"{test_m['recall'] * 100:.1f}%",
        'Test F1-Score': f"{test_m['f1_score']:.3f}",
        'Test ROC-AUC': f"{test_m['roc_auc']:.3f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

## 2. Confusion Matrices (Held-Out Test Set, N=61)
Visualizing true positives, false positives, true negatives, and false negatives for CAD and individual coronary vessels.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
targets_list = ['cad', 'lad', 'lcx', 'rca']
titles = ['Cath (Overall CAD)', 'LAD Stenosis', 'LCX Stenosis', 'RCA Stenosis']

for idx, (target_key, title) in enumerate(zip(targets_list, titles)):
    cm = np.array(metrics_data[target_key]['held_out_test_metrics']['confusion_matrix_labels_0_then_1'])
    ax = axes[idx]
    im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Positive'], fontsize=10)
    ax.set_yticklabels(['Normal', 'Positive'], fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('Ground Truth', fontsize=10)
    
    # Annotate counts inside cells
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Receiver Operating Characteristic (ROC) Curves
Evaluating discriminator sensitivity across variable probability thresholds for all four arterial models.

In [ ]:
# Load raw dataset and test split to generate ROC curves
df_raw = pd.read_excel(DATA_PATH, sheet_name='Sheet 1 - Table 1')
LEAKAGE_COLS = ['Cath', 'LAD', 'LCX', 'RCA']
feature_cols = [c for c in df_raw.columns if c not in LEAKAGE_COLS]
X_all = df_raw[feature_cols]

plt.figure(figsize=(9, 6.5))
colors = {'cad': '#0284c7', 'lad': '#dc2626', 'lcx': '#16a34a', 'rca': '#d97706'}
target_labels = {'cad': 'Cath (CAD)', 'lad': 'LAD Stenosis', 'lcx': 'LCX Stenosis', 'rca': 'RCA Stenosis'}

for target_key in targets_list:
    target_col = metrics_data[target_key]['target']
    y_clean = df_raw[target_col].astype(str).str.strip().str.lower().map({'cad': 1, 'stenotic': 1, 'normal': 0}).astype(int)
    
    # Re-split using identical seed
    X_train, X_test, y_train, y_test = train_test_split(X_all, y_clean, test_size=0.20, random_state=42, stratify=y_clean)
    
    model_path = MODEL_DIR / f"{target_key}_model.joblib"
    model = joblib.load(model_path)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    
    plt.plot(fpr, tpr, color=colors[target_key], lw=2.2, label=f"{target_labels[target_key]} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], color='#94a3b8', linestyle='--', lw=1.5, label='Chance Baseline (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
plt.title('CardioVision3D — Held-Out Test Set ROC Curves (N=61)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Cross-Validation Candidate Benchmarks (CV ROC-AUC)
Comparison of candidate models (Logistic Regression vs. Random Forest Ensemble) across 5 stratified folds.

In [ ]:
cv_records = []
for target_key in targets_list:
    candidates = metrics_data[target_key]['cross_validation_candidates']
    for model_name, stats in candidates.items():
        cv_records.append({
            'Target': target_labels[target_key],
            'Algorithm': model_name.replace('_', ' ').title(),
            'Mean ROC-AUC': stats['cv_roc_auc_mean'],
            'Std Dev': stats['cv_roc_auc_std']
        })

df_cv = pd.DataFrame(cv_records)

# Bar plot comparison
plt.figure(figsize=(10, 5))
targets_unique = list(target_labels.values())
x = np.arange(len(targets_unique))
width = 0.35

lr_scores = df_cv[df_cv['Algorithm'] == 'Logistic Regression']['Mean ROC-AUC'].values
rf_scores = df_cv[df_cv['Algorithm'] == 'Random Forest']['Mean ROC-AUC'].values

plt.bar(x - width/2, lr_scores, width, label='Logistic Regression', color='#38bdf8', edgecolor='#0284c7')
plt.bar(x + width/2, rf_scores, width, label='Random Forest Ensemble', color='#0284c7', edgecolor='#0f172a')

plt.ylabel('5-Fold Cross-Validation ROC-AUC', fontsize=11)
plt.title('Algorithm Comparison: Logistic Regression vs Random Forest Ensemble', fontsize=12, fontweight='bold')
plt.xticks(x, targets_unique, fontsize=10)
plt.ylim(0.5, 1.0)
plt.legend(loc='lower right', fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Detailed Classification Reports
Precision, recall, F1-score, and support broken down by diagnostic class.

In [ ]:
for target_key in targets_list:
    print('=' * 65)
    print(f"TARGET: {target_labels[target_key]} | Selected: {metrics_data[target_key]['selected_model']}")
    print('=' * 65)
    clf_rep = metrics_data[target_key]['held_out_test_metrics']['classification_report']
    print(pd.DataFrame(clf_rep).T.round(3))
    print('\n')